# World Cup Agent Arena — Gemini Build

Same 8-step loop as the sample notebook, but powered by **Google Gemini** (`gemini-2.0-flash`) instead of Anthropic.

| Step | What it does |
|------|--------------|
| Setup | Keys, endpoints, Gemini client |
| 1 | Find fixtures and pick one |
| 2 | Fetch + digest Sportmonks pre-match data |
| 3 | Fetch + digest Polymarket live prices |
| 4 | Fetch + digest Supabase historical stats |
| 5 | Predict the result (agent's independent view) |
| 6 | Decide whether to bet (edge vs market) |
| 7 | Place the order (or skip) |
| 8 | Record everything to the Reasoning Ledger |

**Before running:** fill in `ARENA_KEY` and `GEMINI_API_KEY` in the Setup cell.

## Setup — keys, endpoints, Gemini client

In [ ]:
import os, json, time, uuid, re, requests
from google import genai
from google.genai import types

# ── Arena endpoints ────────────────────────────────────────────────────────────
ARENA            = "https://staging.stair-ai.com"
SPORTMONKS_PROXY = f"{ARENA}/api/v1/data/proxy/sportmonks/v3/football"
POLYMARKET_CLOB  = f"{ARENA}/api/v1/data/proxy/polymarket-clob"
POLYMARKET_GAMMA = f"{ARENA}/api/v1/data/proxy/polymarket-gamma"

# ── Credentials  ──────────────────────────────────────────────────────────────
ARENA_KEY      = "your_arena_key_here"       # https://staging.stair-ai.com/api-keys
GEMINI_API_KEY = "your_gemini_api_key_here"

# Shared read-only Supabase key (same for every builder on staging)
SUPABASE     = "https://ezvbmtvrvzageqixvdak.supabase.co"
SUPABASE_KEY = "sb_publishable__m8bOkD05ToFwATpaWST5w_2-3fGS7V"

# ── Auth headers ───────────────────────────────────────────────────────────────
H_ARENA  = {"x-api-key": ARENA_KEY}
H_PUBLIC = {"apikey": SUPABASE_KEY}
H_WCA    = {"apikey": SUPABASE_KEY, "Accept-Profile": "world_cup_arena"}

# ── Tournament constant ────────────────────────────────────────────────────────
SPORTMONKS_SEASON_ID = 26618   # World Cup 2026

# ── Gemini model + settings ────────────────────────────────────────────────────
GEMINI_MODEL    = "gemini-3.5-flash"
LLM_MAX_TOKENS  = 2400

# Gemini thinking config — include_thoughts exposes the reasoning trace
THINKING_CFG = types.ThinkingConfig(
    include_thoughts=True,
    thinking_budget=1024,
)

# ── Gemini client ──────────────────────────────────────────────────────────────
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

# ── Ledger schema version ──────────────────────────────────────────────────────
LEDGER_SCHEMA_VERSION = "0.3"

# ── Helper: extract (final_text, thinking_text) from a Gemini response ─────────
def _extract(resp):
    """Return (final_text, thinking_text) from a Gemini generate_content response.
    Parts with part.thought=True are the internal chain-of-thought;
    the remaining text parts form the final answer."""
    thinking_parts, answer_parts = [], []
    for part in resp.candidates[0].content.parts:
        text = getattr(part, "text", None)
        if not text:
            continue
        if getattr(part, "thought", False):
            thinking_parts.append(text)
        else:
            answer_parts.append(text)
    return "\n".join(answer_parts), "\n\n".join(thinking_parts)

# ── Helper: build a model_invocation dict for the ledger ──────────────────────
def _mi(resp):
    """Build the ModelInvocation sub-object for a Gemini ledger record."""
    _, thinking = _extract(resp)
    um = resp.usage_metadata
    mi = {
        "provider":   "gemini",
        "model_name": GEMINI_MODEL,
        "tokens_in":  getattr(um, "prompt_token_count", None),
        "tokens_out": getattr(um, "candidates_token_count", None),
    }
    if thinking:
        mi["internal_reasoning"] = thinking
    return mi

# ── Helper: call Gemini with a system prompt ───────────────────────────────────
def gemini_call(system_prompt: str, user_content: str):
    """Single generate_content call with system instruction + thinking enabled."""
    return gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=user_content,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            max_output_tokens=LLM_MAX_TOKENS,
            thinking_config=THINKING_CFG,
        ),
    )

# ── Helper: pull JSON object from raw LLM text ─────────────────────────────────
def _parse_json(raw: str):
    """Extract the first {...} block from raw text and parse it as JSON."""
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    return json.loads(m.group(0)) if m else None

# ── Sanity check ───────────────────────────────────────────────────────────────
_missing = [n for n, v in [("ARENA_KEY", ARENA_KEY), ("GEMINI_API_KEY", GEMINI_API_KEY)]
            if "FILL IN" in v]
if _missing:
    print(f"WARNING: still need to set: {', '.join(_missing)}")
else:
    print("Both keys are set. ✓")
print(f"Arena  : {ARENA}")
print(f"Model  : {GEMINI_MODEL}")
print(f"Season : World Cup 2026 (id {SPORTMONKS_SEASON_ID})")
print("Setup complete — run cells below in order.")

Both keys are set. ✓
Arena  : https://staging.stair-ai.com
Model  : gemini-3.5-flash
Season : World Cup 2026 (id 26618)
Setup complete — run cells below in order.


## Step 1 · Find the matches (fixtures) and pick one

We call the arena's Sportmonks **proxy** for the WC2026 season schedule.
The proxy wraps the upstream reply in an envelope `{body, statusCode, …}` — we peel `body → data`.

We hard-code the tournament opener **Mexico vs South Africa** (`fixture_id 19609127`) to match the sample notebook.

In [9]:
r = requests.get(
    f"{SPORTMONKS_PROXY}/schedules/seasons/{SPORTMONKS_SEASON_ID}",
    headers=H_ARENA, timeout=10,
)
r.raise_for_status()

envelope = r.json()
schedule = envelope["body"]["data"]

print(f"HTTP {r.status_code} OK")
print(f"Envelope keys: {list(envelope.keys())}")
print(f"Schedule entries: {len(schedule)}\n")

# Hard-code the opener for this walkthrough
SPORTMONKS_FIXTURE_ID = 19609127
print(f"Chosen fixture: Mexico vs South Africa (fixture_id {SPORTMONKS_FIXTURE_ID})")

HTTP 200 OK
Envelope keys: ['body', 'duration', 'headers', 'requestId', 'statusCode', '_proxy']
Schedule entries: 7

Chosen fixture: Mexico vs South Africa (fixture_id 19609127)


In [10]:
# Resolve the Polymarket event slug for this fixture
r = requests.get(
    f"{ARENA}/api/v1/web/mapping",
    params={"fixture_id": SPORTMONKS_FIXTURE_ID},
    headers=H_ARENA, timeout=10,
)
r.raise_for_status()
mappings = r.json().get("mappings") or []
polymarket_event_slug = mappings[0]["polymarket_event_slug"] if mappings else None

print(f"HTTP {r.status_code} OK")
if polymarket_event_slug:
    print(f"Polymarket event slug: {polymarket_event_slug!r}")
else:
    print("No Polymarket mapping for this fixture — predict-only mode.")

HTTP 200 OK
Polymarket event slug: 'fifwc-mex-rsa-2026-06-11'


## Step 2 · Fetch the match's pre-game data, then summarize it with Gemini

We pull the fixture detail from Sportmonks (model predictions, bookmaker odds, expected goals)
then ask Gemini to boil it into a compact JSON digest for later steps.

In [23]:
r = requests.get(
    f"{SPORTMONKS_PROXY}/fixtures/{SPORTMONKS_FIXTURE_ID}",
    params={"include": "participants;predictions;odds;xgfixture"},
    headers=H_ARENA, timeout=15,
)
r.raise_for_status()

fixture = r.json()["body"]["data"]
home = next(p for p in fixture["participants"] if p["meta"]["location"] == "home")
away = next(p for p in fixture["participants"] if p["meta"]["location"] == "away")

print(f"HTTP {r.status_code} OK")
print(f"Fixture : {fixture['name']}")
print(f"Kickoff : {fixture.get('starting_at')}")
print(f"Home    : {home['name']} ({home['short_code']})")
print(f"Away    : {away['name']} ({away['short_code']})")
print(f"  Predictions : {len(fixture.get('predictions') or [])} rows")
print(f"  Odds        : {len(fixture.get('odds') or [])} rows")
print(f"  xG          : {len(fixture.get('xgfixture') or [])} rows")

HTTP 200 OK
Fixture : Mexico vs South Africa
Kickoff : 2026-06-11 19:00:00
Home    : Mexico (MEX)
Away    : South Africa (ZAF)
  Predictions : 28 rows
  Odds        : 2182 rows
  xG          : 0 rows


In [32]:
import pandas as pd
from IPython.display import display

# ── Fixture header ────────────────────────────────────────────
print(f"\n{'='*50}")
print(f"  {fixture['name']}")
print(f"  Kickoff: {fixture.get('starting_at')}  |  ID: {fixture['id']}")
print(f"{'='*50}\n")

# ── Participants ──────────────────────────────────────────────
participants_df = pd.DataFrame([
    {"Role": "Home", "Team": home["name"], "Short code": home["short_code"], "Team ID": home["id"]},
    {"Role": "Away", "Team": away["name"], "Short code": away["short_code"], "Team ID": away["id"]},
])
print("── Participants ──")
display(participants_df)

# ── ML Predictions ────────────────────────────────────────────
preds = fixture.get("predictions") or []
pred_rows = []
for p in preds:
    probs = p.get("predictions") or {}
    if "home" in probs or "draw" in probs:
        pred_rows.append({
            "type_id":   p.get("type_id"),
            "home prob": round(float(probs.get("home", 0)), 4),
            "draw prob": round(float(probs.get("draw", 0)), 4),
            "away prob": round(float(probs.get("away", 0)), 4),
        })

print("\n── ML Predictions (sample) ──")
if pred_rows:
    display(pd.DataFrame(pred_rows[:5]))
else:
    print("no rows")

# ── Bookmaker Odds ────────────────────────────────────────────
odds = fixture.get("odds") or []
odds_1x2 = [o for o in odds if o.get("market_id") == 1]
odds_rows = []
for o in sorted(odds_1x2, key=lambda x: x.get("latest_bookmaker_update") or "", reverse=True)[:10]:
    prob_raw = (o.get("probability") or "0").replace("%", "").strip()
    odds_rows.append({
        "bookmaker_id": o.get("bookmaker_id"),
        "label":        o.get("label"),
        "decimal odds": float(o.get("value") or 0),
        "implied prob": f"{round(float(prob_raw), 1)}%",
        "updated":      (o.get("latest_bookmaker_update") or "")[:16],
    })

print("\n── Bookmaker Odds (1X2, latest 10) ──")
if odds_rows:
    display(pd.DataFrame(odds_rows))
else:
    print("no rows")

# ── xG ────────────────────────────────────────────────────────
xg = fixture.get("xgfixture") or []
print("\n── Expected Goals (xG) ──")
if xg:
    display(pd.DataFrame([{"participant_id": x.get("participant_id"), "xG": x.get("value")} for x in xg]))
else:
    print("no data on staging")


  Mexico vs South Africa
  Kickoff: 2026-06-11 19:00:00  |  ID: 19609127

── Participants ──


,Role,Team,Short code,Team ID
0,Home,Mexico,MEX,18576
1,Away,South Africa,ZAF,18555



── ML Predictions (sample) ──


,type_id,home prob,draw prob,away prob
0,233,28.40,46.08,25.52
1,237,40.84,27.78,31.38
2,238,50.35,9.54,40.11



── Bookmaker Odds (1X2, latest 10) ──


,bookmaker_id,label,decimal odds,implied prob,updated
0,2,Draw,4.20,23.8%,2026-06-06 04:21
1,2,Home,1.44,69.2%,2026-06-06 04:21
2,2,Away,7.50,13.3%,2026-06-06 04:21
3,28,Home,1.44,69.4%,2026-06-06 04:15
4,28,Draw,4.33,23.1%,2026-06-06 04:15
5,28,Away,7.75,12.9%,2026-06-06 04:15
6,58,Home,1.41,70.9%,2026-06-06 04:15
7,58,Draw,4.30,23.3%,2026-06-06 04:15
8,58,Away,8.20,12.2%,2026-06-06 04:15
9,13,Home,1.44,69.4%,2026-06-06 04:15



── Expected Goals (xG) ──
no data on staging


In [12]:
DIGEST_SYS = (
    "You are a soccer analyst. You receive a raw Sportmonks pre-match payload for "
    "one fixture and must distil it into a self-contained JSON digest that a "
    "downstream LLM (with no other context about Sportmonks) will read.\n\n"
    "## Output schema (return ONLY this JSON — no prose, no code fences)\n"
    "{\n"
    "  'fixture'                       : str,\n"
    "  'home_team'                     : str,\n"
    "  'away_team'                     : str,\n"
    "  'sportmonks_ml_win_prob'        : {home_code: float, 'draw': float, away_code: float} | null,\n"
    "  'bookmaker_consensus_win_prob'  : {home_code: float, 'draw': float, away_code: float} | null,\n"
    "  'bookmaker_count'               : int | null,\n"
    "  'expected_goals'                : {home_code: float, away_code: float} | null,\n"
    "  'data_availability': {\n"
    "    'sportmonks_ml'       : 'available' | 'missing',\n"
    "    'bookmaker_consensus' : 'available' | 'missing',\n"
    "    'expected_goals'      : 'available' | 'missing'\n"
    "  },\n"
    "  'summary': str\n"
    "}\n\n"
    "Use null (not 0) when source data is missing. Do NOT fabricate values."
)

digest_input = json.dumps({
    "fixture":     fixture["name"],
    "home_code":   home["short_code"],
    "away_code":   away["short_code"],
    "predictions": fixture.get("predictions"),
    "odds":        sorted(
        [o for o in (fixture.get("odds") or []) if o.get("market_id") == 1],
        key=lambda o: o.get("latest_bookmaker_update") or "",
    )[-1:],
    "xGFixture":   fixture.get("xgfixture"),
})

llm_digest = gemini_call(DIGEST_SYS, digest_input)
raw_digest, thinking_digest = _extract(llm_digest)
sportmonks_digest = _parse_json(raw_digest)

print(f"Gemini reasoned for {len(thinking_digest)} chars before answering.")
print("Sportmonks digest:\n")
print(json.dumps(sportmonks_digest, indent=2))

Gemini reasoned for 1673 chars before answering.
Sportmonks digest:

{
  "fixture": "Mexico vs South Africa",
  "home_team": "Mexico",
  "away_team": "South Africa",
  "sportmonks_ml_win_prob": {
    "MEX": 40.84,
    "draw": 27.78,
    "ZAF": 31.38
  },
  "bookmaker_consensus_win_prob": null,
  "bookmaker_count": 1,
  "expected_goals": null,
  "data_availability": {
    "sportmonks_ml": "available",
    "bookmaker_consensus": "missing",
    "expected_goals": "missing"
  },
  "summary": "Sportmonks ML models favor Mexico with a 40.84% win probability, while South Africa is given a 31.38% chance, and a draw stands at 27.78%. Bookmaker consensus and expected goals (xG) data are unavailable due to incomplete records."
}


## Step 3 · Fetch the Polymarket betting market + summarize it

- **Gamma** → event + 3 child markets (condition ids + YES/NO token ids)
- **CLOB** → live mid price per YES token (= implied probability)
- Gemini digests the result into implied win probabilities + execution handles.

In [33]:
import re as _re
TICKER_RE = _re.compile(r"^fifwc-([a-z]{2,4})-([a-z]{2,4})-(\d{4}-\d{2}-\d{2})$")

def _clob_mid(token_id_str: str):
    if not token_id_str:
        return None
    try:
        resp = requests.get(
            f"{POLYMARKET_CLOB}/midpoint",
            params={"token_id": token_id_str},
            headers=H_ARENA, timeout=10,
        )
        if not resp.ok:
            return None
        body = resp.json().get("body")
        if isinstance(body, dict) and "mid" in body:
            return float(body["mid"])
    except Exception:
        pass
    return None

def _outcome_from_slug(market_slug, ticker, home_code, away_code):
    if not market_slug.startswith(ticker + "-"):
        return None
    suffix = market_slug[len(ticker) + 1:]
    if suffix == home_code: return "home"
    if suffix == "draw":    return "draw"
    if suffix == away_code: return "away"
    return None

moneyline = None

if polymarket_event_slug:
    r = requests.get(
        f"{POLYMARKET_GAMMA}/events",
        params={"slug": polymarket_event_slug},
        headers=H_ARENA, timeout=15,
    )
    r.raise_for_status()
    events = r.json().get("body") or []
    event  = events[0] if events else None

    if event:
        ticker = (event.get("ticker") or "").lower()
        m = TICKER_RE.match(ticker)
        if m:
            pm_home, pm_away, _ = m.groups()
            outcomes = {}
            for mkt in (event.get("markets") or []):
                key = _outcome_from_slug((mkt.get("slug") or "").lower(),
                                         ticker, pm_home, pm_away)
                if key is None:
                    continue
                try:
                    token_ids = json.loads(mkt.get("clobTokenIds") or "[]")
                except json.JSONDecodeError:
                    token_ids = []
                token_yes = token_ids[0] if token_ids else None
                outcomes[key] = {
                    "team_code":       key if key == "draw" else (
                                           pm_home.upper() if key == "home" else pm_away.upper()),
                    "condition_id":    mkt.get("conditionId"),
                    "token_yes":       token_yes,
                    "current_mid_yes": _clob_mid(token_yes),
                }
            moneyline = {
                "sportmonks_match_id":   SPORTMONKS_FIXTURE_ID,
                "fixture":               event.get("title"),
                "kickoff_utc":           event.get("startDate"),
                "polymarket_event_slug": polymarket_event_slug,
                "outcomes":              outcomes,
            }

if moneyline is None:
    print("No tradable Polymarket market — predict-only mode.")
else:
    n = sum(1 for o in moneyline["outcomes"].values() if o["current_mid_yes"] is not None)
    print(f"Moneyline built for: {moneyline['fixture']}")
    print(f"Live mids: {n}/3 outcomes")
    print(json.dumps(moneyline, indent=2, default=str))

Moneyline built for: Mexico vs. South Africa
Live mids: 3/3 outcomes
{
  "sportmonks_match_id": 19609127,
  "fixture": "Mexico vs. South Africa",
  "kickoff_utc": "2026-04-06T22:48:47.767995Z",
  "polymarket_event_slug": "fifwc-mex-rsa-2026-06-11",
  "outcomes": {
    "home": {
      "team_code": "MEX",
      "condition_id": "0x4cd77d456c83e7d8c569a8fb8f6396c3f40154f657e6d970733e2b1b6a7110ff",
      "token_yes": "20779063998268474490699884714808310289659170477959115489741275295270359962039",
      "current_mid_yes": 0.685
    },
    "draw": {
      "team_code": "draw",
      "condition_id": "0x0a4b9beb6128863db2b107f185521597a426356f1d9a23c7001401edfd32014b",
      "token_yes": "11634144673803325466643188834337300341803737096067092239834760823912726000274",
      "current_mid_yes": 0.205
    },
    "away": {
      "team_code": "RSA",
      "condition_id": "0x17dfc75726fa95d4054d91e80295c8b3e494569617e67a7e620e27562b7952b0",
      "token_yes": "115307860962719805060784163204351769077176

In [35]:
if moneyline:
    # ── Event header ──────────────────────────────────────────
    print(f"\n{'='*50}")
    print(f"  {moneyline['fixture']}")
    print(f"  Kickoff : {moneyline['kickoff_utc']}")
    print(f"  Slug    : {moneyline['polymarket_event_slug']}")
    print(f"{'='*50}\n")

    # ── Moneyline outcomes table ──────────────────────────────
    outcomes_rows = []
    for position, (key, o) in enumerate(moneyline["outcomes"].items()):
        mid = o["current_mid_yes"]
        outcomes_rows.append({
            "position":     key,            # home / draw / away
            "team_code":    o["team_code"],
            "mid price":    mid,
            "implied prob": f"{round(mid * 100, 1)}%" if mid else "N/A",
            "token_yes":    (o["token_yes"] or "")[:20] + "...",   # truncate — 78 digit number
            "condition_id": (o["condition_id"] or "")[:12] + "...",
        })

    print("── Moneyline outcomes ──")
    display(pd.DataFrame(outcomes_rows))

    # ── Probability sanity check ──────────────────────────────
    mids = [o["current_mid_yes"] for o in moneyline["outcomes"].values() if o["current_mid_yes"]]
    total = sum(mids)
    print(f"\n── Probability sum check ──")
    display(pd.DataFrame([{
        "home":  f"{mids[0]*100:.1f}%" if len(mids) > 0 else "N/A",
        "draw":  f"{mids[1]*100:.1f}%" if len(mids) > 1 else "N/A",
        "away":  f"{mids[2]*100:.1f}%" if len(mids) > 2 else "N/A",
        "sum":   round(total, 4),
        "gap to 1.0": round(1 - total, 4),
        "status": "✓ healthy" if 0.95 <= total <= 1.05 else "⚠ stale prices",
    }]))
else:
    print("No moneyline — nothing to visualize.")


  Mexico vs. South Africa
  Kickoff : 2026-04-06T22:48:47.767995Z
  Slug    : fifwc-mex-rsa-2026-06-11

── Moneyline outcomes ──


,position,team_code,mid price,implied prob,token_yes,condition_id
0,home,MEX,0.685,68.5%,20779063998268474490...,0x4cd77d456c...
1,draw,draw,0.205,20.5%,11634144673803325466...,0x0a4b9beb61...
2,away,RSA,0.105,10.5%,11530786096271980506...,0x17dfc75726...



── Probability sum check ──


,home,draw,away,sum,gap to 1.0,status
0,68.5%,20.5%,10.5%,0.995,0.005,✓ healthy


In [14]:
POLYMARKET_DIGEST_SYS = (
    "You are an analyst digesting a Polymarket moneyline (3-way match-winner) "
    "market response into a self-contained JSON for a downstream LLM.\n\n"
    "## Output schema (return ONLY this JSON — no prose, no code fences)\n"
    "{\n"
    "  'fixture'           : str,\n"
    "  'market_handle'     : str,\n"
    "  'implied_win_prob'  : {home_code: float, 'draw': float, away_code: float} | null,\n"
    "  'sum_implied_prob'  : float | null,\n"
    "  'execution_handles' : {home_code: {condition_id, token_yes}, 'draw': {...}, away_code: {...}},\n"
    "  'data_availability' : 'mids_available' | 'mids_partial' | 'mids_missing' | 'no_market',\n"
    "  'summary'           : str\n"
    "}\n\nUse null when input is null. Do NOT fabricate prices."
)

if moneyline is None:
    polymarket_digest = {
        "fixture": None, "market_handle": None, "implied_win_prob": None,
        "sum_implied_prob": None, "execution_handles": None,
        "data_availability": "no_market",
        "summary": f"No Polymarket moneyline for fixture {SPORTMONKS_FIXTURE_ID}.",
    }
else:
    llm_pm = gemini_call(POLYMARKET_DIGEST_SYS, json.dumps(moneyline))
    raw_pm, thinking_pm = _extract(llm_pm)
    polymarket_digest = _parse_json(raw_pm)
    print(f"Gemini digested the market ({len(thinking_pm)} chars of thinking).")

print("\nMarket digest:\n")
print(json.dumps(polymarket_digest, indent=2))

Gemini digested the market (1146 chars of thinking).

Market digest:

{
  "fixture": "Mexico vs. South Africa",
  "market_handle": "fifwc-mex-rsa-2026-06-11",
  "implied_win_prob": {
    "MEX": 0.685,
    "draw": 0.205,
    "RSA": 0.105
  },
  "sum_implied_prob": 0.995,
  "execution_handles": {
    "MEX": {
      "condition_id": "0x4cd77d456c83e7d8c569a8fb8f6396c3f40154f657e6d970733e2b1b6a7110ff",
      "token_yes": "20779063998268474490699884714808310289659170477959115489741275295270359962039"
    },
    "draw": {
      "condition_id": "0x0a4b9beb6128863db2b107f185521597a426356f1d9a23c7001401edfd32014b",
      "token_yes": "11634144673803325466643188834337300341803737096067092239834760823912726000274"
    },
    "RSA": {
      "condition_id": "0x17dfc75726fa95d4054d91e80295c8b3e494569617e67a7e620e27562b7952b0",
      "token_yes": "115307860962719805060784163204351769077176612029040401546976102705811910754396"
    }
  },
  "data_availability": "mids_available",
  "summary": "Polymarket

## Step 4 · Fetch historical team stats from Supabase + summarize

⚠️ Team IDs differ between systems: Sportmonks uses 458/146 for Mexico/South Africa; the Supabase stats DB uses **country_id 147/211** (StatsBomb-derived).

Sub-steps: 4a discover catalog → 4b fetch rows → 4c Gemini digest.

In [15]:
# 4a · Discover what tables exist
r = requests.get(
    f"{SUPABASE}/rest/v1/catalog_full",
    params={"select": "table_name,category,row_count,table_description",
            "order":  "category,table_name"},
    headers=H_PUBLIC, timeout=10,
)
r.raise_for_status()
catalog = r.json()

print(f"HTTP {r.status_code} OK — catalog:\n")
for t in catalog:
    desc = (t.get("table_description") or "-").replace("\n", " ")[:60]
    cat  = t.get("category") or "?"
    print(f"  [{cat:11s}] {t['table_name']:30s}  rows={t['row_count']:>5d}  - {desc}")

WANTED_TABLE = "ads_a_country_style"
print(f"\nUsing: {WANTED_TABLE}")

HTTP 200 OK — catalog:

  [checkpoint ] d_checkpoint_minutes            rows=  130  - Records the actual end minute and period for each checkpoint
  [checkpoint ] d_checkpoint_runs               rows=  130  - One row per checkpoint processing job run, capturing the exe
  [checkpoint ] d_checkpoint_snapshot           rows=  260  - One row per team per match checkpoint (HT, FT, ET1, ET2) cap
  [checkpoint ] d_match_scores                  rows=  206  - One row per match checkpoint records the final or halftime s
  [dimension  ] dim_checkpoint                  rows=    4  - One row per checkpoint in a match timeline, containing the c
  [dimension  ] dim_match                       rows=   65  - Dimension table storing one row per football match, containi
  [priors     ] ads_a_country_struct            rows=   66  - Country-level prior data capturing each nation's most recent
  [priors     ] ads_a_country_style             rows=   71  - One row per country containing aggregated set-piece s

In [16]:
# 4b · Fetch rows for both teams
TEAM_A_ID = 147   # Mexico
TEAM_B_ID = 211   # South Africa

r = requests.get(
    f"{SUPABASE}/rest/v1/{WANTED_TABLE}",
    params={"country_id": f"in.({TEAM_A_ID},{TEAM_B_ID})", "select": "*"},
    headers=H_WCA, timeout=10,
)
r.raise_for_status()
priors_rows = r.json()

print(f"HTTP {r.status_code} OK — {len(priors_rows)} row(s)\n")
print(json.dumps(priors_rows, indent=2, default=str))

HTTP 200 OK — 2 row(s)

[
  {
    "country_id": 147,
    "set_piece_shots": 94,
    "set_piece_goals": 2,
    "conversion_rate": 0.0212765957446809,
    "group_matches": 9,
    "group_goals_against": 8,
    "ko_matches": 1,
    "ko_goals_against": 2,
    "group_gpg": 0.888888888888889,
    "ko_gpg": 2
  },
  {
    "country_id": 211,
    "set_piece_shots": 26,
    "set_piece_goals": 2,
    "conversion_rate": 0.0769230769230769,
    "group_matches": 6,
    "group_goals_against": 14,
    "ko_matches": 1,
    "ko_goals_against": 2,
    "group_gpg": 2.33333333333333,
    "ko_gpg": 2
  }
]


In [17]:
# 4c · Gemini digest of the stats
SUPABASE_DIGEST_SYS = (
    "You are an analyst aggregating Supabase priors data for one fixture into "
    "a self-contained JSON digest for a downstream LLM.\n\n"
    "## Output schema (return ONLY this JSON — no prose, no code fences)\n"
    "{\n"
    "  'fixture'      : str,\n"
    "  'source_table' : str,\n"
    "  'teams': {\n"
    "    home_code: {\n"
    "      'set_piece_efficiency' : float | null,\n"
    "      'set_piece_sample'     : int   | null,\n"
    "      'group_goals_per_game' : float | null,\n"
    "      'ko_goals_per_game'    : float | null\n"
    "    },\n"
    "    away_code: { same shape }\n"
    "  },\n"
    "  'data_availability': 'rich' | 'partial' | 'sparse',\n"
    "  'summary': str\n"
    "}\n\nUse null when input is empty/missing. Don't fabricate values."
)

sb_input = json.dumps({
    "fixture":      fixture["name"],
    "source_table": WANTED_TABLE,
    "home_code":    home["short_code"],
    "away_code":    away["short_code"],
    "home_id":      TEAM_A_ID,
    "away_id":      TEAM_B_ID,
    "rows":         priors_rows,
}, default=str)

llm_sb = gemini_call(SUPABASE_DIGEST_SYS, sb_input)
raw_sb, thinking_sb = _extract(llm_sb)
supabase_digest = _parse_json(raw_sb)

print(f"Gemini summarized the stats ({len(thinking_sb)} chars of thinking).")
print("Stats digest:\n")
print(json.dumps(supabase_digest, indent=2))

Gemini summarized the stats (1046 chars of thinking).
Stats digest:

{
  "fixture": "Mexico vs South Africa",
  "source_table": "ads_a_country_style",
  "teams": {
    "MEX": {
      "set_piece_efficiency": 0.0212765957446809,
      "set_piece_sample": 94,
      "group_goals_per_game": 0.888888888888889,
      "ko_goals_per_game": 2.0
    },
    "ZAF": {
      "set_piece_efficiency": 0.0769230769230769,
      "set_piece_sample": 26,
      "group_goals_per_game": 2.33333333333333,
      "ko_goals_per_game": 2.0
    }
  },
  "data_availability": "rich",
  "summary": "Mexico has a substantial historical set-piece sample size of 94 shots but a low conversion rate of 2.13%, contrasting with South Africa's smaller sample size of 26 shots converted at a higher rate of 7.69%. Defensively (or offensively via goals per game/against metrics depending on interpretation, here 'gpg' goals against context), South Africa has shown a higher concession rate in group matches (2.33 goals per game) compare

## Step 5 · Predict the result (Gemini's independent view)

Gemini combines the Sportmonks + Supabase digests into a prediction **without seeing the market** — this independence is where edge comes from.

In [18]:
PREDICT_SYS = (
    "You are a soccer match analyst. You receive two pre-distilled digests for "
    "one fixture and must produce the agent's own outcome prediction.\n\n"
    "## Output schema (return ONLY this JSON — no prose, no code fences)\n"
    "{\n"
    "  'fixture'         : str,\n"
    "  'outcome'         : str,          // home_code | 'draw' | away_code\n"
    "  'probability'     : float,        // 0..1; confidence in outcome\n"
    "  'rationale'       : str,          // 1-3 sentences naming the teams + signals used\n"
    "  'used_signals': {\n"
    "    'sportmonks' : 'leaned_on' | 'unavailable',\n"
    "    'supabase'   : 'leaned_on' | 'unavailable'\n"
    "  },\n"
    "  'confidence_level': 'high' | 'medium' | 'low'\n"
    "}\n\n"
    "Do NOT consult the market. Probability must reflect only the priors."
)

predict_input = json.dumps({
    "fixture":           fixture["name"],
    "home_code":         home["short_code"],
    "away_code":         away["short_code"],
    "sportmonks_digest": sportmonks_digest,
    "supabase_digest":   supabase_digest,
})

llm_predict = gemini_call(PREDICT_SYS, predict_input)
raw_pred, thinking_pred = _extract(llm_predict)
prediction = _parse_json(raw_pred)

print(f"Gemini formed its prediction ({len(thinking_pred)} chars of thinking):\n")
print(json.dumps(prediction, indent=2))
if prediction:
    print(f"\n→ Most likely '{prediction['outcome']}' "
          f"at {prediction['probability']:.0%} ({prediction['confidence_level']})")

Gemini formed its prediction (1078 chars of thinking):

{
  "fixture": "Mexico vs South Africa",
  "outcome": "MEX",
  "probability": 0.41,
  "rationale": "Mexico is favored by the Sportmonks ML model with a 40.84% probability, supported by a much tighter group-stage defensive record (0.89 goals conceded per game) compared to South Africa's leakier average of 2.33.",
  "used_signals": {
    "sportmonks": "leaned_on",
    "supabase": "leaned_on"
  },
  "confidence_level": "medium"
}

→ Most likely 'MEX' at 41% (medium)


## Step 6 · Decide whether to bet (edge vs market)

**Edge** = agent's probability − market's implied probability.
Positive edge → market underprices → go **long**. Negative → market overprices → go **short**. Small edge → skip.

In [19]:
STRATEGY_SYS = (
    "You are a bankroll manager for a $100 demo account. You receive the agent's "
    "own prediction and the current Polymarket market view, and decide whether "
    "to trade and on what terms.\n\n"
    "## How to decide\n"
    "  Edge = prediction.probability − polymarket_digest.implied_win_prob[prediction.outcome]\n"
    "  |edge| < 5pp  → don't trade (noise)\n"
    "  |edge| 5-15pp → $1-2 (modest)\n"
    "  |edge| > 15pp → $3-5 (high conviction; cap $5)\n"
    "  Halve size if confidence_level is 'low'. Skip if data_availability != 'mids_available'.\n\n"
    "## Output schema (return ONLY this JSON — no prose, no code fences)\n"
    "{\n"
    "  'should_trade'  : bool,\n"
    "  'outcome'       : str,\n"
    "  'direction'     : 'long' | 'short',\n"
    "  'size_usdc'     : float,\n"
    "  'limit_price'   : float,\n"
    "  'edge_pp'       : float,\n"
    "  'market_handle' : str,\n"
    "  'rationale'     : str\n"
    "}\n\nBe conservative: small wallet, weak conviction → skipping is valid."
)

strategy_input = json.dumps({
    "prediction":        prediction,
    "polymarket_digest": polymarket_digest,
})

llm_strategy = gemini_call(STRATEGY_SYS, strategy_input)
raw_strat, thinking_strat = _extract(llm_strategy)
strategy = _parse_json(raw_strat)

print(f"Strategy decision ({len(thinking_strat)} chars of thinking):\n")
print(json.dumps(strategy, indent=2))
if strategy and strategy.get("should_trade"):
    print(f"\n→ {strategy['direction'].upper()} ${strategy['size_usdc']:.2f} "
          f"on '{strategy['outcome']}' (edge {strategy['edge_pp']:+.1f}pp)")
elif strategy:
    print(f"\n→ No trade — edge {strategy.get('edge_pp', 0):+.1f}pp isn't worth it.")

Strategy decision (1221 chars of thinking):

{
  "should_trade": true,
  "outcome": "MEX",
  "direction": "short",
  "size_usdc": 5.0,
  "limit_price": 0.685,
  "edge_pp": -27.5,
  "market_handle": "fifwc-mex-rsa-2026-06-11",
  "rationale": "The agent predicts a 41% chance of Mexico winning, whereas the market has it heavily inflated at 68.5%. This represents a massive negative edge of -27.5pp, making a short position on Mexico highly favorable."
}

→ SHORT $5.00 on 'MEX' (edge -27.5pp)


## Step 7 · Place the bet (open a position)

If Step 6 decided to trade, we POST an order to the arena. An **idempotency key** (UUID) prevents accidental double-submission.

The `/orders` endpoint may still 404 on staging — that's expected. The payload is printed either way.

In [20]:
order_payload  = None
order_response = None

if strategy and strategy.get("should_trade"):
    if strategy.get("direction") == "long":
        team_code = strategy["outcome"]
    else:
        # Short: buy the cheaper alternative outcome instead
        alts = {k: v for k, v in
                (polymarket_digest.get("implied_win_prob") or {}).items()
                if k != strategy["outcome"]}
        team_code = min(alts, key=alts.get) if alts else None

    if team_code:
        order_payload = {
            "fixture_code":          str(SPORTMONKS_FIXTURE_ID),
            "team_code":             team_code,
            "usd_size":              1,
            "limit_price":           strategy["limit_price"],
            "time_in_force_seconds": 30,
            "idempotency_key":       str(uuid.uuid4()),
        }
        print("Strategy says TRADE. Order payload:\n")
        print(json.dumps(order_payload, indent=2))

        try:
            r = requests.post(
                f"{ARENA}/api/v1/arena/orders",
                headers=H_ARENA, timeout=30,
                json=order_payload,
            )
            if r.ok:
                order_response = r.json()
                print(f"\nHTTP {r.status_code} OK — order submitted:")
                print(json.dumps(order_response, indent=2))
            elif r.status_code == 404:
                print(f"\nHTTP 404 — orders endpoint not yet live on staging (expected).")
            else:
                print(f"\nHTTP {r.status_code} — rejected: {r.text[:300]}")
        except Exception as e:
            print(f"\nOrder POST failed: {type(e).__name__}: {e}")
    else:
        print("Short direction but couldn't find an alternative outcome — skipping order.")
else:
    print("Strategy said no trade — skipping order step.")

Strategy says TRADE. Order payload:

{
  "fixture_code": "19609127",
  "team_code": "RSA",
  "usd_size": 1,
  "limit_price": 0.685,
  "time_in_force_seconds": 30,
  "idempotency_key": "5536bcbc-7cb6-4c26-b33c-9b0bb09135e0"
}

HTTP 400 — rejected: {"defined":false,"code":"BAD_REQUEST","status":400,"message":"Input validation failed","data":{"issues":[{"expected":"string","code":"invalid_type","path":["usd_size"],"message":"Invalid input"}]}}


## Step 8 · Record the agent's reasoning to the Ledger

Every step is recorded as a typed **ledger record** so the arena can audit and score the agent.

Record types:
- **Observing** — data the agent read (API calls, DB queries)
- **Thinking** — LLM reasoning calls
- **Acting** — decisions the agent took (prediction, order)

Records are submitted as a single batch to `/api/v1/arena/ledger/records/batch`.

In [21]:
SESSION_ID = str(uuid.uuid4())   # one session per agent run

def _new_record(behavior: str, **fields) -> dict:
    return {
        "schema_version": LEDGER_SCHEMA_VERSION,
        "record_id":      str(uuid.uuid4()),
        "session_id":     SESSION_ID,
        "behavior":       behavior,
        "timestamp":      time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        **fields,
    }

def _trunc(obj, limit=30000):
    s = obj if isinstance(obj, str) else json.dumps(obj, default=str)
    return s if len(s) <= limit else s[:limit] + f"…[truncated, was {len(s)} chars]"

# (1) Trigger
rec_trigger = _new_record(
    "Observing",
    trigger_type="cron",
    trigger_description="Synthetic cron trigger — agent woke to analyze WC2026 fixtures.",
)

# (2) Schedule fetch
rec_sm_schedule = _new_record(
    "Observing",
    upstream_record_id=[rec_trigger["record_id"]],
    description=f"Fetched WC2026 schedule from Sportmonks proxy. Chose fixture {SPORTMONKS_FIXTURE_ID}.",
    source="sportmonks_proxy",
)

# (3) Polymarket slug lookup
rec_pm_slug = _new_record(
    "Observing",
    upstream_record_id=[rec_sm_schedule["record_id"]],
    description=f"Resolved Polymarket slug: {polymarket_event_slug!r}",
    source="arena_mapping",
)

# (4) Sportmonks fixture detail
rec_sm_fixture = _new_record(
    "Observing",
    upstream_record_id=[rec_sm_schedule["record_id"]],
    description=f"Fetched fixture detail for {SPORTMONKS_FIXTURE_ID} (predictions, odds, xG).",
    source="sportmonks_proxy",
)

# (5) Gemini digest — Sportmonks
rec_th_sportmonks = _new_record(
    "Thinking",
    upstream_record_id=[rec_sm_fixture["record_id"]],
    model_invocation=_mi(llm_digest),
    prompt=_trunc(DIGEST_SYS, limit=16000),
    inputs=[{"input_record_id": rec_sm_fixture["record_id"],
             "input_payload":   _trunc(digest_input)}],
    output_payload=_trunc(sportmonks_digest),
)

# (6) Polymarket event + mids fetch
rec_pm_event = _new_record(
    "Observing",
    upstream_record_id=[rec_pm_slug["record_id"]],
    description="Fetched Polymarket Gamma event + CLOB mids for all 3 outcomes.",
    source="polymarket_gamma+clob",
)

# (7) Gemini digest — Polymarket
rec_th_polymarket = _new_record(
    "Thinking",
    upstream_record_id=[rec_pm_event["record_id"]],
    model_invocation=(_mi(llm_pm) if moneyline else {"provider": "gemini", "model_name": GEMINI_MODEL}),
    prompt=_trunc(POLYMARKET_DIGEST_SYS, limit=16000),
    inputs=[{"input_record_id": rec_pm_event["record_id"],
             "input_payload":   _trunc(moneyline or {})}],
    output_payload=_trunc(polymarket_digest),
)

# (8) Supabase catalog
rec_sb_catalog = _new_record(
    "Observing",
    upstream_record_id=[rec_trigger["record_id"]],
    description="Fetched Supabase catalog_full to discover available tables.",
    source="supabase",
)

# (9) Supabase priors fetch
rec_sb_priors = _new_record(
    "Observing",
    upstream_record_id=[rec_sb_catalog["record_id"]],
    description=f"Fetched {WANTED_TABLE} for country_ids {TEAM_A_ID},{TEAM_B_ID}.",
    source="supabase",
)

# (10) Gemini digest — Supabase
rec_th_supabase = _new_record(
    "Thinking",
    upstream_record_id=[rec_sb_priors["record_id"]],
    model_invocation=_mi(llm_sb),
    prompt=_trunc(SUPABASE_DIGEST_SYS, limit=16000),
    inputs=[{"input_record_id": rec_sb_priors["record_id"],
             "input_payload":   _trunc(sb_input)}],
    output_payload=_trunc(supabase_digest),
)

# (11) Gemini thinking — Prediction
rec_th_predict = _new_record(
    "Thinking",
    upstream_record_id=[rec_th_sportmonks["record_id"], rec_th_supabase["record_id"]],
    model_invocation=_mi(llm_predict),
    prompt=_trunc(PREDICT_SYS, limit=16000),
    inputs=[
        {"input_record_id": rec_th_sportmonks["record_id"], "input_payload": _trunc(sportmonks_digest)},
        {"input_record_id": rec_th_supabase["record_id"],   "input_payload": _trunc(supabase_digest)},
    ],
    output_payload=_trunc(prediction),
)

# (12) Acting — Prediction
_pred_prob = max(0.001, min(0.999, float(prediction["probability"])))
rec_act_predict = _new_record(
    "Acting",
    upstream_record_id=[rec_th_predict["record_id"]],
    action_type=     "prediction",
    target_system=   "arena",
    action_summary=  f"Predict {prediction['outcome']} @ p={_pred_prob:.2f} for fixture {SPORTMONKS_FIXTURE_ID}",
    parameters=      {
        "fixture_code": str(SPORTMONKS_FIXTURE_ID),
        "outcome":      prediction["outcome"],
        "probability":  _pred_prob,
    },
    dry_run=False,
    execution_status="confirmed",
)

# (13) Gemini thinking — Strategy
rec_th_strategy = _new_record(
    "Thinking",
    upstream_record_id=[rec_th_predict["record_id"], rec_th_polymarket["record_id"]],
    model_invocation=_mi(llm_strategy),
    prompt=_trunc(STRATEGY_SYS, limit=16000),
    inputs=[
        {"input_record_id": rec_th_predict["record_id"],    "input_payload": _trunc(prediction)},
        {"input_record_id": rec_th_polymarket["record_id"], "input_payload": _trunc(polymarket_digest)},
    ],
    output_payload=_trunc(strategy),
)

records = [
    rec_trigger, rec_sm_schedule, rec_pm_slug, rec_pm_event,
    rec_sm_fixture, rec_th_sportmonks, rec_th_polymarket,
    rec_sb_catalog, rec_sb_priors, rec_th_supabase,
    rec_th_predict, rec_act_predict, rec_th_strategy,
]

# (14) Acting — Order (only if we actually submitted one)
if strategy and strategy.get("should_trade") and order_payload:
    submitted_ok = isinstance(order_response, dict) and bool(order_response)
    rec_act_order = _new_record(
        "Acting",
        upstream_record_id=[rec_th_strategy["record_id"]],
        action_type=     "open_order",
        target_system=   "arena",
        action_summary=  (f"Open {strategy['direction']} ${strategy['size_usdc']:.2f} on "
                          f"{strategy['outcome']} @ ≤{strategy['limit_price']}"),
        parameters=      order_payload,
        dry_run=False,
        execution_status="pending" if submitted_ok else "failed",
        execution_id=    (order_response.get("order_id") if submitted_ok else None),
    )
    records.append(rec_act_order)

print(f"Built {len(records)} ledger records:\n")
for rec in records:
    label = (rec.get("description")
             or rec.get("action_summary")
             or rec.get("trigger_description")
             or rec.get("prompt", "")[:50])
    print(f"  {rec['behavior']:12s}  {rec['record_id'][:8]}…  {label}")

Built 14 ledger records:

  Observing     a40a5b48…  Synthetic cron trigger — agent woke to analyze WC2026 fixtures.
  Observing     42cdd202…  Fetched WC2026 schedule from Sportmonks proxy. Chose fixture 19609127.
  Observing     31ba1c30…  Resolved Polymarket slug: 'fifwc-mex-rsa-2026-06-11'
  Observing     02b28d9b…  Fetched Polymarket Gamma event + CLOB mids for all 3 outcomes.
  Observing     cd478851…  Fetched fixture detail for 19609127 (predictions, odds, xG).
  Thinking      f3f56b02…  You are a soccer analyst. You receive a raw Sportm
  Thinking      293e3aa8…  You are an analyst digesting a Polymarket moneylin
  Observing     a7f587a6…  Fetched Supabase catalog_full to discover available tables.
  Observing     e2dc17a2…  Fetched ads_a_country_style for country_ids 147,211.
  Thinking      120b84b1…  You are an analyst aggregating Supabase priors dat
  Thinking      ee9cc031…  You are a soccer match analyst. You receive two pr
  Acting        16ed9cde…  Predict MEX @ p=0.41 

In [22]:
# Submit the batch to the arena ledger
try:
    r = requests.post(
        f"{ARENA}/api/v1/arena/ledger/records/batch",
        headers=H_ARENA, timeout=60,
        json={"records": records},
    )
    if r.status_code == 404:
        print(f"HTTP 404 — ledger endpoint not yet live on staging (expected).\n"
              f"The {len(records)} records above are exactly what a real run would submit.")
    elif r.ok:
        resp = r.json()
        print(f"HTTP {r.status_code} OK — ledger accepted: "
              f"{len(resp.get('records', []))} stored, {len(resp.get('errors', []))} error(s).")
        for e in resp.get("errors", []):
            print(f"  [#{e.get('index')}] {e.get('code')}: {e.get('message')}")
    else:
        print(f"HTTP {r.status_code} — ledger rejected: {r.text[:300]}")
except Exception as e:
    print(f"Ledger POST failed: {type(e).__name__}: {e}")

HTTP 200 OK — ledger accepted: 1 stored, 13 error(s).
  [#0] ledger_error: Failed to forward record to ledger
  [#1] ledger_error: Failed to forward record to ledger
  [#2] ledger_error: Failed to forward record to ledger
  [#3] ledger_error: Failed to forward record to ledger
  [#4] ledger_error: Failed to forward record to ledger
  [#5] ledger_error: Failed to forward record to ledger
  [#6] ledger_error: Failed to forward record to ledger
  [#7] ledger_error: Failed to forward record to ledger
  [#8] ledger_error: Failed to forward record to ledger
  [#9] ledger_error: Failed to forward record to ledger
  [#10] ledger_error: Failed to forward record to ledger
  [#12] ledger_error: Failed to forward record to ledger
  [#13] ledger_error: Failed to forward record to ledger
